# Analyzing Equity Research with AI_COMPLETE Document Intelligence

This notebook demonstrates how to use Snowflake's **AI_COMPLETE** function to process equity research PDFs directly from a Snowflake stage — no parsing, no text extraction pipelines, no intermediate tables needed.

With AI_COMPLETE's document intelligence capability (GA April 2026), you can point a model directly at PDF files and ask questions, extract structured data, compare documents, and analyze charts.

**What we'll cover:**
1. Exploring research reports stored on a Snowflake stage
2. Single-document Q&A directly from PDFs
3. Chart and visualization analysis
4. Structured data extraction with guaranteed JSON output
5. Multi-document comparison and cross-referencing
6. Batch processing all documents at scale
7. Building an analytics-ready table with visualizations

## Setup

We'll work with the `FSI_DEMO_DB.EQUITY_RESEARCH` schema which contains equity research PDFs stored on a Snowflake internal stage with server-side encryption (required for AI_COMPLETE document processing).

In [ ]:
%%sql -r setup_context
USE DATABASE FSI_DEMO_DB;
USE SCHEMA EQUITY_RESEARCH;
USE WAREHOUSE AICOLLEGE;

## 1. Explore the Research Reports on Stage

Our equity research PDFs are stored in the `@REPORTS` internal stage. The stage uses **server-side encryption** (`SNOWFLAKE_SSE`), which is required for AI_COMPLETE document processing.

Let's see what's available using the `DIRECTORY()` table function.

In [ ]:
%%sql -r stage_files
SELECT 
    RELATIVE_PATH,
    ROUND(SIZE / 1024, 1) AS size_kb,
    LAST_MODIFIED
FROM DIRECTORY(@REPORTS)
ORDER BY RELATIVE_PATH

## 2. Single-Document Q&A

The simplest use case: point AI_COMPLETE at a PDF and ask a question. No parsing needed — the model reads the document directly.

**Key syntax:**
```sql
AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT('Your question about {0}', TO_FILE('@stage', 'file.pdf'))
)
```

- `TO_FILE('@stage', 'filename.pdf')` creates a FILE object reference
- `PROMPT()` combines your text instructions with the file reference
- `{0}`, `{1}`, etc. are positional placeholders for each `TO_FILE` argument

In [ ]:
%%sql -r single_doc_qa
-- Ask about key risks in a JPMorgan earnings review
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'What are the key risks mentioned in this research note? Provide a concise bulleted summary. {0}',
        TO_FILE('@REPORTS', '2026-01-15_jpm_earnings_review.pdf')
    )
) AS key_risks

In [ ]:
%%sql -r nvidia_analysis
-- Ask about competitive dynamics in NVIDIA's earnings review
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Based on this research note, what is the competitive landscape for NVIDIA? What threats does the analyst identify? {0}',
        TO_FILE('@REPORTS', '2026-01-28_nvda_earnings_review.pdf')
    )
) AS competitive_analysis

## 3. Chart and Visualization Analysis

AI_COMPLETE can read charts, graphs, and diagrams embedded in PDFs. Let's first look at what the model is working with — here's a page from the Goldman Sachs report showing the charts we'll analyze.

In [ ]:
%%sql -r pdf_url
-- Generate a presigned URL so we can render the PDF inline
SELECT GET_PRESIGNED_URL(@REPORTS, 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf', 3600) AS url

In [ ]:
import requests
import fitz  # PyMuPDF
from IPython.display import display, Image as IPImage
from io import BytesIO

# Download the PDF and render pages with charts
url = pdf_url['URL'][0]
pdf_bytes = requests.get(url).content
doc = fitz.open(stream=pdf_bytes, filetype='pdf')
doc_pages = [doc[i] for i in range(len(doc))]  # store for later use

# Render pages that contain charts (pages 2 and 4 based on AI_COMPLETE's analysis)
for page_num in [1, 3]:  # 0-indexed
    page = doc_pages[page_num]
    pix = page.get_pixmap(matrix=fitz.Matrix(2, 2))  # 2x zoom for clarity
    img_bytes = pix.tobytes('png')
    print(f"\n--- Page {page_num + 1} ---")
    display(IPImage(data=img_bytes))

Now let's use AI_COMPLETE to analyze these charts. We'll extract structured metadata about each chart using `response_format`, then display the AI's analysis side-by-side with the rendered pages.

In [ ]:
%%sql -r chart_inventory
-- Extract structured chart metadata with guaranteed JSON
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Identify every chart and visualization in this document. For each one, describe the chart type, what data it displays, and the key takeaway. {0}',
        TO_FILE('@REPORTS', 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf')
    ),
    response_format => {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {
                'charts': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'chart_number': {'type': 'integer'},
                            'page': {'type': 'integer'},
                            'chart_type': {'type': 'string'},
                            'title': {'type': 'string'},
                            'data_description': {'type': 'string'},
                            'key_takeaway': {'type': 'string'}
                        },
                        'required': ['chart_number', 'page', 'chart_type', 'title', 'data_description', 'key_takeaway']
                    }
                }
            },
            'required': ['charts']
        }
    }
) AS charts_json

In [ ]:
import json
import textwrap
import matplotlib.pyplot as plt
from matplotlib.offsetbox import OffsetImage, AnnotationBbox
from PIL import Image as PILImage
from io import BytesIO

# Parse the structured chart analysis
charts = json.loads(chart_inventory['CHARTS_JSON'][0])['charts']

# Display each chart: PDF page on left, AI analysis on right
for chart_info in charts[:2]:
    page_idx = chart_info['page'] - 1  # 0-indexed
    page = doc_pages[page_idx] if page_idx < len(doc_pages) else None
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6), gridspec_kw={'width_ratios': [1, 1]})
    
    # Left: PDF page render
    if page is not None:
        pix = page.get_pixmap(matrix=fitz.Matrix(1.5, 1.5))
        img = PILImage.open(BytesIO(pix.tobytes('png')))
        axes[0].imshow(img)
    axes[0].set_title(f"PDF Page {chart_info['page']}", fontsize=12, fontweight='bold')
    axes[0].axis('off')
    
    # Right: AI analysis
    axes[1].axis('off')
    analysis_text = (
        f"Chart {chart_info['chart_number']}: {chart_info['title']}\n"
        f"{'=' * 50}\n\n"
        f"Type: {chart_info['chart_type']}\n\n"
        f"Data:\n{textwrap.fill(chart_info['data_description'], width=55)}\n\n"
        f"Key Takeaway:\n{textwrap.fill(chart_info['key_takeaway'], width=55)}"
    )
    axes[1].text(0.05, 0.95, analysis_text, transform=axes[1].transAxes,
                fontsize=10, verticalalignment='top', fontfamily='monospace',
                bbox=dict(boxstyle='round', facecolor='#f0f4f8', alpha=0.8))
    axes[1].set_title("AI_COMPLETE Analysis", fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

In [ ]:
%%sql -r chart_data_extraction
-- Use structured outputs to extract chart data as guaranteed JSON
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Look at the chart showing hyperscaler capital expenditure spending in this document. Extract the approximate annual capex values for each year shown. {0}',
        TO_FILE('@REPORTS', 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf')
    ),
    response_format => {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {
                'chart_title': {'type': 'string'},
                'data_points': {
                    'type': 'array',
                    'items': {
                        'type': 'object',
                        'properties': {
                            'year': {'type': 'integer'},
                            'capex_billions': {'type': 'number'}
                        }
                    }
                }
            },
            'required': ['chart_title', 'data_points']
        }
    }
) AS capex_data

In [ ]:
import json
import matplotlib.pyplot as plt

# Parse the structured JSON from the previous cell
capex_json = json.loads(chart_data_extraction['CAPEX_DATA'][0])

years = [d['year'] for d in capex_json['data_points']]
values = [d['capex_billions'] for d in capex_json['data_points']]

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(years, values, color='#29B5E8', edgecolor='white', width=0.7)
ax.set_xlabel('Year')
ax.set_ylabel('Capital Expenditure ($B)')
ax.set_title(f"Hyperscaler Capex — Extracted from PDF Chart via AI_COMPLETE")
ax.set_xticks(years)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

for bar, val in zip(bars, values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'${val:.0f}B', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r chart_qa
-- Ask questions that require reading both charts and text together
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Looking at the S&P 500 forecast chart and the surrounding text in this document:
         1. What is the year-end 2026 price target for the S&P 500?
         2. What EPS growth assumptions underpin this forecast?
         3. What are the biggest risks to this forecast according to the document? {0}',
        TO_FILE('@REPORTS', 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf')
    )
) AS forecast_analysis

In [ ]:
%%sql -r demographics_chart
-- Analyze charts in the Global Economics aging report
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'This document discusses global aging and its economic impact. Summarize the key findings from the charts and data visualizations. What is the positive story about global aging that the authors present? {0}',
        TO_FILE('@REPORTS', 'Global Economics Analyst_ The Path to 2075 \u2014 The Positive Story of Global Aging (Daly_Njie_Allen).pdf')
    )
) AS aging_analysis

## 4. Structured Data Extraction with `response_format`

Use AI_COMPLETE's **structured outputs** feature to guarantee valid JSON responses. By providing a `response_format` with a JSON schema, the model is constrained to produce output matching your exact schema — no regex cleanup, no `TRY_PARSE_JSON` needed.

In [ ]:
%%sql -r structured_extraction
-- Extract structured metadata with guaranteed JSON schema
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Extract metadata from this equity research note. {0}',
        TO_FILE('@REPORTS', '2026-02-05_lly_initiation.pdf')
    ),
    response_format => {
        'type': 'json',
        'schema': {
            'type': 'object',
            'properties': {
                'ticker': {'type': 'string'},
                'company_name': {'type': 'string'},
                'analyst_name': {'type': 'string'},
                'publish_date': {'type': 'string'},
                'note_type': {'type': 'string'},
                'rating': {'type': 'string'},
                'rating_change': {'type': 'string'},
                'conviction': {'type': 'string'},
                'price_target': {'type': 'string'},
                'key_thesis': {'type': 'string'}
            },
            'required': ['ticker', 'company_name', 'analyst_name', 'publish_date', 'note_type', 'rating', 'rating_change', 'conviction', 'price_target', 'key_thesis']
        }
    }
) AS extracted_json

## 5. Multi-Document Comparison

AI_COMPLETE can process **multiple documents in a single prompt** (up to 5 for Claude models, up to 20 for Gemini). This enables cross-document analysis without any ETL.

In [ ]:
%%sql -r semi_comparison
-- Compare NVIDIA vs AMD: two semiconductor research notes
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Compare these two semiconductor research notes. Which company does the analyst prefer and why? Summarize the key differences in outlook, valuation, and risk.
         Document 1: {0}
         Document 2: {1}',
        TO_FILE('@REPORTS', '2026-01-28_nvda_earnings_review.pdf'),
        TO_FILE('@REPORTS', '2026-01-29_amd_earnings_review.pdf')
    )
) AS semiconductor_comparison

In [ ]:
%%sql -r cross_doc_charts
-- Cross-reference: Goldman Sachs market forecast + NVIDIA earnings
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'Document 1 is a macro equity market forecast with charts showing S&P 500 targets and hyperscaler capex. Document 2 is an NVIDIA earnings review. How does NVIDIA''s performance connect to the broader market thesis? Specifically: how does the hyperscaler capex chart relate to NVIDIA''s revenue outlook?
         Document 1: {0}
         Document 2: {1}',
        TO_FILE('@REPORTS', 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf'),
        TO_FILE('@REPORTS', '2026-01-28_nvda_earnings_review.pdf')
    )
) AS macro_micro_connection

In [ ]:
%%sql -r healthcare_compare
-- Compare healthcare notes: bullish UNH vs bearish PFE (same analyst)
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'These are two healthcare research notes. Compare the investment thesis for each company. What makes one attractive while the other is concerning? What sector themes emerge?
         Note 1: {0}
         Note 2: {1}',
        TO_FILE('@REPORTS', '2026-02-10_unh_sector_update.pdf'),
        TO_FILE('@REPORTS', '2026-02-12_pfe_risk_alert.pdf')
    )
) AS healthcare_comparison

## 6. Batch Processing with Structured Outputs

Process every PDF on the stage using `DIRECTORY()` + `AI_COMPLETE` + `response_format`. The structured outputs feature guarantees valid JSON from every document — no post-processing needed.

In [ ]:
%%sql -r batch_extraction
-- Extract structured metadata from every research report with guaranteed JSON
SELECT 
    d.RELATIVE_PATH AS file_name,
    PARSE_JSON(
        AI_COMPLETE(
            MODEL => 'claude-opus-4-6',
            PROMPT => PROMPT(
                'Extract metadata from this research document. {0}',
                TO_FILE('@REPORTS', d.RELATIVE_PATH)
            ),
            response_format => {
                'type': 'json',
                'schema': {
                    'type': 'object',
                    'properties': {
                        'ticker': {'type': 'string'},
                        'company_name': {'type': 'string'},
                        'analyst_name': {'type': 'string'},
                        'publish_date': {'type': 'string'},
                        'note_type': {'type': 'string'},
                        'rating': {'type': 'string'},
                        'rating_change': {'type': 'string'},
                        'conviction': {'type': 'string'},
                        'price_target': {'type': 'string'},
                        'key_thesis': {'type': 'string'}
                    },
                    'required': ['ticker', 'company_name', 'analyst_name', 'publish_date', 'note_type', 'rating', 'rating_change', 'conviction', 'price_target', 'key_thesis']
                }
            }
        )
    ) AS extracted_data
FROM DIRECTORY(@REPORTS) d
WHERE d.RELATIVE_PATH LIKE '%.pdf'
ORDER BY d.RELATIVE_PATH

## 7. Build an Analytics-Ready Table

Flatten the extracted JSON into a proper relational table — transforming raw PDFs into queryable structured data.

In [ ]:
%%sql -r create_analytics_table
CREATE OR REPLACE TABLE EXTRACTED_RESEARCH_NOTES AS
WITH raw_extractions AS (
    SELECT 
        d.RELATIVE_PATH AS file_name,
        PARSE_JSON(
            AI_COMPLETE(
                MODEL => 'claude-opus-4-6',
                PROMPT => PROMPT(
                    'Extract metadata from this research document. {0}',
                    TO_FILE('@REPORTS', d.RELATIVE_PATH)
                ),
                response_format => {
                    'type': 'json',
                    'schema': {
                        'type': 'object',
                        'properties': {
                            'ticker': {'type': 'string'},
                            'company_name': {'type': 'string'},
                            'analyst_name': {'type': 'string'},
                            'publish_date': {'type': 'string'},
                            'note_type': {'type': 'string'},
                            'rating': {'type': 'string'},
                            'rating_change': {'type': 'string'},
                            'conviction': {'type': 'string'},
                            'price_target': {'type': 'string'},
                            'key_thesis': {'type': 'string'}
                        },
                        'required': ['ticker', 'company_name', 'analyst_name', 'publish_date', 'note_type', 'rating', 'rating_change', 'conviction', 'price_target', 'key_thesis']
                    }
                }
            )
        ) AS extracted
    FROM DIRECTORY(@REPORTS) d
    WHERE d.RELATIVE_PATH LIKE '%.pdf'
)
SELECT
    file_name,
    extracted:ticker::STRING AS ticker,
    extracted:company_name::STRING AS company_name,
    extracted:analyst_name::STRING AS analyst_name,
    TRY_TO_DATE(extracted:publish_date::STRING) AS publish_date,
    extracted:note_type::STRING AS note_type,
    extracted:rating::STRING AS rating,
    extracted:rating_change::STRING AS rating_change,
    extracted:conviction::STRING AS conviction,
    extracted:price_target::STRING AS price_target,
    extracted:key_thesis::STRING AS key_thesis,
    CURRENT_TIMESTAMP() AS extracted_at
FROM raw_extractions

In [ ]:
%%sql -r view_extracted
SELECT ticker, company_name, analyst_name, publish_date, note_type,
       rating, rating_change, conviction, price_target, key_thesis
FROM EXTRACTED_RESEARCH_NOTES
ORDER BY publish_date

## 8. Visualize the Extracted Data

In [ ]:
%%sql -r ratings_summary
SELECT 
    rating_change,
    COUNT(*) AS num_notes,
    LISTAGG(ticker, ', ') WITHIN GROUP (ORDER BY publish_date) AS tickers
FROM EXTRACTED_RESEARCH_NOTES
WHERE ticker IS NOT NULL
GROUP BY rating_change
ORDER BY num_notes DESC

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Build visualization from extracted data
df = view_extracted.copy()

# Color-code by rating change
colors = {'Downgrade': '#E74C3C', 'Maintain': '#3498DB', 'Upgrade': '#2ECC71', 'Initiation': '#9B59B6'}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Chart 1: Rating changes by ticker
ax1 = axes[0]
tickers = df[df['TICKER'].notna()]['TICKER'].tolist()
rating_changes = df[df['TICKER'].notna()]['RATING_CHANGE'].tolist()
bar_colors = [colors.get(rc, '#95A5A6') for rc in rating_changes]
ax1.barh(tickers, range(len(tickers)), color=bar_colors)
ax1.set_xlabel('')
ax1.set_title('Rating Actions by Ticker')
ax1.set_xticks([])
for i, (ticker, rc) in enumerate(zip(tickers, rating_changes)):
    ax1.text(0.1, i, f'{rc}', va='center', fontsize=10, fontweight='bold')
ax1.spines['top'].set_visible(False)
ax1.spines['right'].set_visible(False)
ax1.spines['bottom'].set_visible(False)

# Chart 2: Conviction distribution
ax2 = axes[1]
conviction_counts = df[df['CONVICTION'].notna()]['CONVICTION'].value_counts()
conv_colors = {'High': '#27AE60', 'Medium': '#F39C12', 'Low': '#E74C3C'}
wedge_colors = [conv_colors.get(c, '#95A5A6') for c in conviction_counts.index]
ax2.pie(conviction_counts.values, labels=conviction_counts.index, colors=wedge_colors,
        autopct='%1.0f%%', startangle=90, textprops={'fontsize': 11})
ax2.set_title('Analyst Conviction Distribution')

plt.tight_layout()
plt.show()

In [ ]:
%%sql -r portfolio_summary
-- Use AI_COMPLETE to generate a portfolio-level summary across multiple documents
SELECT AI_COMPLETE(
    MODEL => 'claude-opus-4-6',
    PROMPT => PROMPT(
        'You are a portfolio strategist. Based on these research notes, provide: 1. Sectors to OVERWEIGHT (with reasoning) 2. Sectors to UNDERWEIGHT (with reasoning) 3. Key risks to monitor. Be concise — 3-4 bullet points per category.
         Document 1 (macro forecast): {0}
         Document 2 (tech): {1}
         Document 3 (healthcare): {2}
         Document 4 (energy): {3}',
        TO_FILE('@REPORTS', 'US Stocks Are Forecast to Rise 6% in 2026 _ Goldman Sachs.pdf'),
        TO_FILE('@REPORTS', '2026-01-28_nvda_earnings_review.pdf'),
        TO_FILE('@REPORTS', '2026-02-10_unh_sector_update.pdf'),
        TO_FILE('@REPORTS', '2026-02-18_xom_risk_alert.pdf')
    )
) AS portfolio_positioning

## 9. Model Comparison

| Model | Context Window | Max Pages | Docs/Prompt | Best For |
|-------|---------------|-----------|-------------|----------|
| `claude-opus-4-6` | 200K tokens | 100 | 5 | Highest accuracy, complex reasoning |
| `claude-sonnet-4-6` | 1M tokens | 100 | 5 | Large context + Claude quality |
| `gemini-3.1-pro` | 1M tokens | 3,000 | 20 | Very large documents (up to 37.5MB) |

In [ ]:
%%sql -r model_comparison
-- Compare models on the same structured extraction task
SELECT 
    'claude-opus-4-6' AS model,
    AI_COMPLETE(
        MODEL => 'claude-opus-4-6',
        PROMPT => PROMPT(
            'What is the rating, price target, and one-sentence investment thesis? Be concise. {0}',
            TO_FILE('@REPORTS', '2026-02-05_lly_initiation.pdf')
        )
    ) AS response
UNION ALL
SELECT 
    'gemini-3.1-pro' AS model,
    AI_COMPLETE(
        MODEL => 'gemini-3.1-pro',
        PROMPT => PROMPT(
            'What is the rating, price target, and one-sentence investment thesis? Be concise. {0}',
            TO_FILE('@REPORTS', '2026-02-05_lly_initiation.pdf')
        )
    ) AS response

## Summary

**What we demonstrated:**

| Capability | How |
|-----------|-----|
| Direct PDF processing | `TO_FILE('@stage', 'file.pdf')` inside `PROMPT()` |
| Chart/graph analysis | Model reads visual elements natively from PDFs |
| Guaranteed JSON output | `response_format` with JSON schema — no regex, no `TRY_PARSE_JSON` |
| Multi-document reasoning | Up to 5 docs per prompt (Claude) or 20 (Gemini) |
| Batch extraction at scale | `DIRECTORY(@stage)` + `AI_COMPLETE` over all files |
| PDF → relational table | One `CREATE TABLE AS` query, zero ETL pipelines |

**Key requirements:**
- Stage must use **server-side encryption** (`ENCRYPTION = (TYPE = 'SNOWFLAKE_SSE')`)
- Supported file types: `.pdf`, `.txt`, `.md` (all models) + `.doc`, `.docx`, `.xls`, `.xlsx`, `.csv`, `.xhtml` (Claude models)
- User must have the `SNOWFLAKE.CORTEX_USER` database role and READ access to the stage